# MDR-TS v12.4
**Temporal-Station Soil Moisture Model + Shallow NN Baseline**

**Author:** Jakob Balkovec
**Affiliation:** Seattle University, Computer Science
**Project:** MDR
**Notebook Type:** Training, Evaluation, Baseline Modeling
**Last Updated:** Sat Feb 14th 2026

---

## Model Summary

- **Model Name:** MDR-TS
- **Version:** v12.4
- **Task:** Regression (Soil Moisture at 5 cm depth)
- **Target Variable:** `soil_moisture_5cm`
- **Temporal Resolution:** Daily

This notebook implements a clean baseline neural model:

1. **Shallow NN Baseline**
   A 2 to 3 layer mlp trained on the derived_new feature set with proper scaling, early stopping, and full evaluation.

Later notebooks may extend this with physics-residual correction and rain-focused modeling, but this notebook must remain a baseline.

---

## Reproducibility

- **Random Seed:** 42
- **Split Directory:** `data/splits/derived_new/`
- **Environment:** MacBook M2 Pro (local kernel)

---

## What’s New

- Stacking v12.1 & v12.3


## 1. Imports + Environment Setup

In [4]:
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

artifact_dir_v12_1 = Path("/Users/jbalkovec/Desktop/MDR/Models/Temporal/v12/v12.1/artifacts")
artifact_dir_v12_3 = Path("/Users/jbalkovec/Desktop/MDR/Models/Temporal/v12/v12.3/artifacts")

v121_path = artifact_dir_v12_1 / "preds_mlp_v12_1.csv"
v123_path = artifact_dir_v12_3 / "preds_mlp_v12_3.csv"  # adjust if named differently

p1 = pd.read_csv(v121_path)
p3 = pd.read_csv(v123_path)

print(len(p1), len(p3))

22720 22720


## 2. Sanity Checks

In [7]:
# check that splits match row-wise
if not (p1["split"].values == p3["split"].values).all():
    raise ValueError("split columns not aligned — need stable join key")

if not np.allclose(p1["y_true"].values, p3["y_true"].values):
    raise ValueError("y_true mismatch — row order differs, cannot stack safely")

print("alignment looks good")

alignment looks good


## 3. Stacking

In [8]:
def compute_metrics(y_true, y_pred):
    return {
        "r2": float(r2_score(y_true, y_pred)),
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "rmse": float(np.sqrt(mean_squared_error(y_true, y_pred))),
    }

merged = pd.DataFrame({
    "split": p1["split"],
    "y_true": p1["y_true"],
    "pred_v12_1": p1["y_pred"],
    "pred_v12_3": p3["y_pred"],
})

val = merged[merged["split"] == "val"]
test = merged[merged["split"] == "test"]

X_val = val[["pred_v12_1", "pred_v12_3"]].values
y_val = val["y_true"].values

X_test = test[["pred_v12_1", "pred_v12_3"]].values
y_test = test["y_true"].values

meta = Ridge(alpha=1.0)
meta.fit(X_val, y_val)

print("stack weights:")
print("intercept:", meta.intercept_)
print("w_v12.1:", meta.coef_[0])
print("w_v12.3:", meta.coef_[1])

val_pred_stack = meta.predict(X_val)
test_pred_stack = meta.predict(X_test)

print("\nval stacked:", compute_metrics(y_val, val_pred_stack))
print("test stacked:", compute_metrics(y_test, test_pred_stack))

stack weights:
intercept: 0.027302040891626966
w_v12.1: 0.2845320664113383
w_v12.3: 0.6640868000866308

val stacked: {'r2': 0.8236598157530103, 'mae': 0.03226849063222814, 'rmse': 0.04229443374727648}
test stacked: {'r2': 0.6895678523512475, 'mae': 0.041064427566069775, 'rmse': 0.05211125881891282}


## 4. Wet | Dry Breakdown (Optional)

In [9]:
test_is_wet = p1.loc[p1["split"] == "test", "is_wet"].values.astype(bool)

print("test wet stacked:", compute_metrics(y_test[test_is_wet], test_pred_stack[test_is_wet]))
print("test dry stacked:", compute_metrics(y_test[~test_is_wet], test_pred_stack[~test_is_wet]))

test wet stacked: {'r2': -1.6934966242989211, 'mae': 0.04181141569830305, 'rmse': 0.04819478449470246}
test dry stacked: {'r2': 0.6847082990611477, 'mae': 0.040963069882932544, 'rmse': 0.05262022403589047}
